<a href="https://colab.research.google.com/github/QuirozAlaniz/Data-Profiling/blob/main/Sesion10_Data_Profiling_Entregable_269723.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:** Francisco Daniel Quiroz Alaniz

**Matrícula:** 269723

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [169]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

#df_netflix.to_csv('netflix_local.csv', index=False)
df_marketing.to_csv('marketing.csv', index=False)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [170]:
# Tu código aquí
print(df_marketing.columns)
df_marketing_renombrado = df_marketing.copy()   #Hacemos la copia de los datos renombrándolo
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower() #Hacemos los títulos de las columnas a minúsculas,
                                                # teniendo en cuenta que debemos ponerle "....columns=..." para que sí afecte la
                                                # los títulos de las columnas.
nombres = {"mntwines":"mnt_wines", "mntfruits":"mnt_fruits",
           "mntmeatproducts":"mnt_meat_products",
           "mntfishproducts":"mnt_fish_products",
           "mntsweetproducts":"mnt_sweet_products",
           "mntgoldprods":"mnt_gold_prods"}     #Hacemos un diccionario con los "nombres_viejos":"nombres_nuevos" para pasárselo
                                                # al método ".rename"
df_marketing_renombrado.rename(columns = nombres, inplace=True)
print(df_marketing_renombrado.columns)

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')
Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kidhome',
       'teenhome', 'dt_customer', 'recency', 'mnt_wines', 'mnt_fruits',
       'mnt_meat_products', 'mnt_fish_products', 'mnt_sweet_products',
       'mnt_gold_prods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'z_costcontact'

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [171]:
# Tu código aquí
print("El número de valores nulos de la columna original es:", df_netflix["date_added"].isnull().sum())
print("El tipo de dato antes de la conversión es:", df_netflix["date_added"].dtype)
df_netflix["date_added"]=pd.to_datetime(df_netflix["date_added"] , format="mixed")
print("El tipo de dato después de la conversión es:", df_netflix["date_added"].dtype)
print("El número de valores nulos de la columna transformada es:",df_netflix["date_added"].isnull().sum())

El número de valores nulos de la columna original es: 10
El tipo de dato antes de la conversión es: object
El tipo de dato después de la conversión es: datetime64[ns]
El número de valores nulos de la columna transformada es: 10


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [172]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [173]:
# Tu código aquí
print("El número de datos duplicados son:", df_marketing_dup.duplicated().sum() )
print("El número de datos duplicados sobre la columna ID es:", df_marketing_dup.duplicated(subset ="ID").sum() )
df_marketing_dup=df_marketing_dup.drop_duplicates()        #Elimina los duplicados dejando la aparición original.
print("El número de datos duplicados después de eliminar los encontrados son:", df_marketing_dup.duplicated(subset ="ID").sum() )

El número de datos duplicados son: 2
El número de datos duplicados sobre la columna ID es: 2
El número de datos duplicados después de eliminar los encontrados son: 0


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [174]:
# Tu código aquí
print(df_netflix.isnull().sum() )  #Como no estamos definiendo ninguna columna en específico,
                                   #se aplica sobre todas las columnas del dataframe
print()
print("Las filas que tienen al menos un valor faltante en cualquier columna son:", df_netflix.isnull().any(axis = 1).sum() )

show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

Las filas que tienen al menos un valor faltante en cualquier columna son: 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [175]:
# Tu código aquí
lista =[]
for columna in df_netflix.columns:                       #iteramos sobre cada columna
  nulos = int(df_netflix[columna].isnull().sum())        #guardamos la cantidad de nulos de la columna en la variable nulos
  numero_filas = len(df_netflix)                         #calculamos el número de filas que tiene el dataset
  valor = ( 1 - nulos /  numero_filas  )*100             #Calculamos el % de nulos de la columna y lo guardamos en la variable valor
  elemento = (valor, columna)                            # hacemos la tupla del (porcentaje, nombre de la columna)
  lista.append(elemento)                                 #Guardamos la tupla en la lista vacía
minimo = min(lista)                                      #obtenemos el valor mínimo de la lista.
print("La columna con menor completitud es la de", minimo[1], "con", minimo[0], "% de completitud" )



La columna con menor completitud es la de director con 69.32066264286631 % de completitud


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [176]:
# Tu código aquí
df_marketing["Marital_Status"].value_counts() #value_counts() es para series, no para dataframes, por eso se aplica a la columna.

,count
Marital_Status,
Married,864
Together,580
Single,480
Divorced,232
Widow,77
Alone,3
Absurd,2
YOLO,2


**Solución**
\
Para este ejercicio vemos que "Alone" significa "solo", lo cual cabría en la categoría "Single", "Absurd" no puede ser clasificado en ninguna otra clasificación, así que la eliminaría al ser solo dos veces la que aparece, y "YOLO" está en la misma situación, no cabe en otra clasificación, se puede eliminar.
\
...aunque siempre he pensado que depende mucho del problema de negocio que se necesite resolver, por eso dicen que la principal habilidad de un analista de datos es "saber del negocio" antes que saber usar herramientas.

---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [177]:
# Tu código aquí
verificacion = df_netflix["show_id"].str.match(r"^s\d+$")
verificacion = verificacion.sum()
print("El procentaje de cumplimiento es:", verificacion/(len(df_netflix))*100, "%" )

El procentaje de cumplimiento es: 100.0 %


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [178]:
#Solución Actividad 8

# Tu código aquí
print(df_marketing["Year_Birth"].describe())
\
print()
\
#Filtramos el dataframe
ordenados = df_marketing["Year_Birth"].sort_values() #Seleccionamos los 10 valores más bajos para ver qué fechas
                                                      #se encuentran ahí. ".sort_values" ordena los valores de una pandas Serie.
print(f'La columna "Year_Birth" ordenada es:\n{ordenados}')

count    2240.000000
mean     1968.805804
std        11.984069
min      1893.000000
25%      1959.000000
50%      1970.000000
75%      1977.000000
max      1996.000000
Name: Year_Birth, dtype: float64

La columna "Year_Birth" ordenada es:
239     1893
339     1899
192     1900
1950    1940
424     1941
        ... 
2213    1995
1850    1995
995     1995
1170    1996
46      1996
Name: Year_Birth, Length: 2240, dtype: int64


In [179]:
#Como podemos ver, hay un salto muy grande desde 1900 a 1940, luego se "compone". Es esta gran distancia la que me hace
#pensar que los primeros tres valores son un error de input, a parte si tuviéramos esas 3 fechas de nacimiento vigentes al día de hoy,
#sería gente de más de cien años.

---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [180]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [181]:
# Paso 1 — ajuste de tipos

#Income debería ser un float pero está como un object
df_practica["Income"] = pd.to_numeric(df_practica["Income"], errors = "coerce") #la función pd.to_numeric pide al menos un argumento,
                                                                                #que es precisamente la columna del dataframe.
                                                                                #La columna la Income original la sustituimos por
                                                                                # la nueva.
print(df_practica.dtypes)

ID                       int64
Year_Birth               int64
Education               object
Marital_Status          object
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer             object
Recency                  int64
MntWines                 int64
MntFruits                int64
MntMeatProducts          int64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds             int64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Complain                 int64
Z_CostContact            int64
Z_Revenue                int64
Response                 int64
dtype: object


In [182]:
# Paso 2 — duplicados
df_practica.duplicated().sum()  #.duplicated() nos da los True o False, los True son los duplicados, mientras que .sum() los cuenta
df_practica = df_practica.drop_duplicates()    #eliminamos todas las filas duplicadas con .drop_duplicates() dejando la aparición
                                               #única.
print(df_practica.duplicated().sum())          #revisamos que ya no haya duplicados

0


In [183]:
# Paso 3 — valores faltantes
df_practica.isnull().sum() #Aplicamos .isnull().sum() a todo el dataframe y nos arroja el nombre de la columna y en seguida
                           #en otra columna el número de valores faltantes.
                           #Podemos observar que solamente la columna "Income" tiene un valor faltante.

,0
ID,0
Year_Birth,0
Education,0
Marital_Status,0
Income,1
Kidhome,0
Teenhome,0
Dt_Customer,0
Recency,0
MntWines,0


In [184]:
# Paso 4 — exploración categórica
df_practica["Marital_Status"].unique() #Aplicamos .unique() a la columna "Marital_Status" y nos arroja los valores únicos.
                                       #Podemos ver que "Together" y "Married" son parecidos, por lo que si se refieren a lo mismo,
                                       #habrá que normalizar.

array(['Together', 'Single', 'Married', 'Divorced'], dtype=object)

**Tu reporte de profiling:**

*(Escribe aquí tu resumen de 3-4 líneas)*
\
El profiling para el primer dataset: df_marketing consistió en normalización de columnas principalmente, para el dataset df_netflix corregimos formatos de fecha que originalmente se tomaba como object. Esto es importante porque hay funciones que toman valores de cierto tipo de dato, por ejemplo para series de tiempo, que, si le pasamos fechas con formato object, no correrá el código. Al final trabajamos con el dataset de práctica para detectar duplicados, eliminarlos, y analizar y decidir qué hacer sobre ciertos tipos de datos que muestra el dataset.